# Langgraph Tutorial

## 1) Activating Models

In [ ]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

c:\Users\deepak.a.dhiman\projects\agentic_ai\.agentic_ai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

True

In [3]:
llm = ChatGroq(model="qwen/qwen3-32b", reasoning_effort=None)
embedding_model= HuggingFaceEmbeddings(model="BAAI/bge-small-en")

In [4]:
text_embedd = embedding_model.embed_query("hi")

In [ ]:
# len(text_embedd)

384

## Creating Vector DB

In [6]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
# from langchain_community.vectorstores import Chroma
from langchain_chroma import Chroma

In [8]:
# loader = PyPDFLoader(file_path=r"C:\Users\deepak.a.dhiman\projects\agentic_ai\AgenticAI_2.0\data\BERT_Paper.pdf")

In [9]:
# document = loader.load()

In [10]:
# len(document)

In [11]:
# splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=150)

In [12]:
# chunks = splitter.split_documents(documents=document)
# len(chunks)

In [8]:
db_path = r"C:\Users\deepak.a.dhiman\projects\agentic_ai\AgenticAI_2.0\3-vectoredb\Chroma"

#intialising chroma collection
# vector_db = Chroma(collection_name="langgraph1", embedding_function=embedding_model, persist_directory=db_path)

In [9]:
#adding document in chroma
# vector_db.add_documents(chunks)

#loading existing chroma collection
vector_db = Chroma(collection_name="langgraph1", embedding_function=embedding_model, persist_directory=db_path)

In [10]:
retreiver = vector_db.as_retriever(search_kwargs={"k":3})

In [11]:
result = retreiver.invoke("who wrote the BERT paper?")

In [12]:
result

[Document(id='27171028-4a0d-43d1-9a8c-1c758d2f9eb8', metadata={'author': '', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.17 (TeX Live 2016) kpathsea version 6.2.2', 'trapped': '/False', 'page': 5, 'creationdate': '2019-05-28T00:07:51+00:00', 'page_label': '6', 'total_pages': 16, 'source': 'C:\\Users\\deepak.a.dhiman\\projects\\agentic_ai\\AgenticAI_2.0\\data\\BERT_Paper.pdf', 'moddate': '2019-05-28T00:07:51+00:00', 'keywords': '', 'producer': 'pdfTeX-1.40.17', 'title': '', 'creator': 'LaTeX with hyperref package', 'subject': ''}, page_content='labels, and we only made a single GLUE evaluation server\nsubmission for each of BERTBASE and BERTLARGE .\n10https://gluebenchmark.com/leaderboard\nWikipedia containing the answer, the task is to\npredict the answer text span in the passage.\nAs shown in Figure 1, in the question answer-\ning task, we represent the input question and pas-\nsage as a single packed sequence, with the ques-\ntion using the A embedding and the pas

## Orchestrating Graph using Langgraph

In [13]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser

In [14]:
# writing Pydantic class

class TopicSelectionParser(BaseModel):
    Topic:str = Field(description="selected topic")
    Reasoning:str = Field(description="Reasoning behind topic selection")

In [20]:
# intialising parser

parser = PydanticOutputParser(pydantic_object=TopicSelectionParser)

In [21]:
#parser instruction

parser.get_format_instructions()

'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"Topic": {"description": "selected topic", "title": "Topic", "type": "string"}, "Reasoning": {"description": "Reasoning behind topic selection", "title": "Reasoning", "type": "string"}}, "required": ["Topic", "Reasoning"]}\n```'